# Recorte de enunciados de PDFs de emestrada — paso a paso

In [1]:
# %pip install pdfplumber pypdf pandas

import re
import os
import shutil
from pathlib import Path
import unicodedata

import pandas as pd
import pdfplumber
from pypdf import PdfReader, PdfWriter, Transformation
from IPython.display import display

In [2]:
root = "/home/daniel/code/clasificador_selectividad"
os.chdir(root)

In [3]:
ANCHO_MIN, ALTO_MIN = 300, 20   # tamaño mínimo para considerar un rectángulo como recuadro de enunciado
MARGEN = 2                      # holgura alrededor del borde al recortar
DPI = 200                       # resolución de los PNG
A4 = (595.28, 841.89)
MARGEN_A4 = 40                  # margen de la hoja de salida
SEPARACION = 15                 # hueco entre recuadros apilados

In [4]:
def detectar_recuadros(page):
    vistos = {}
    for r in page.rects:
        if r["width"] > ANCHO_MIN and r["height"] > ALTO_MIN:
            clave = tuple(round(r[k]) for k in ("x0", "top", "x1", "bottom"))
            vistos[clave] = r
    return sorted(vistos.values(), key=lambda r: r["top"])

In [5]:
PATRON = re.compile(
    r"""
    (?P<asig>SOCIALES[ \t]+(?:II|2)|MATEM[AÁ]TICAS[ \t]+(?:II|2)|QU[IÍ]MICA|F[IÍ]SICA)
    [ \t.,:;-]*
    (?P<anio>(?:19|20)\d{2})
    [ \t.,:;-]*
    (?P<conv>(?:JUNIO|JULIO|SEPTIEMBRE|PONENCIA|RESERVA|MODELO)(?:[ \t]*\d+)?)
    [ \t.,:;-]*
    (?P<ej>[^\n\r]*?)
    [ \t.]*\r?$
    """,
    re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)

In [6]:
def slug(texto):
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"\W+", "_", texto).strip("_").lower()

#print(slug("RESERVA 1"), "|", slug("SEPTIEMBRE"), "|", slug("2023 - Integrales"))

In [7]:
def procesar_pdf(ruta_pdf, output_folder, png=True, dpi=DPI, uno_por_pagina=False):
    filas, recortes = [], []
    png_folder = output_folder / "enunciados_PNG" / f"{ruta_pdf.stem}_png"

    # 1) Detección con pdfplumber
    with pdfplumber.open(ruta_pdf) as pdf:
        for n, page in enumerate(pdf.pages):
            cajas = detectar_recuadros(page)
            for k, r in enumerate(cajas, start=1):
                x0 = max(r["x0"] - MARGEN, 0)
                x1 = min(r["x1"] + MARGEN, page.width)
                top = max(r["top"] - MARGEN, 0)
                bottom = min(r["bottom"] + MARGEN, page.height)

                recorte = page.crop((x0, top, x1, bottom))
                texto_raw = recorte.extract_text() or ""
                texto = texto_raw.replace("\n", " ")
                metadata = PATRON.search(texto_raw)   # buscar con saltos de línea

                if metadata:
                    ej_slug = slug(re.sub(r"EJERCICIO", "", metadata["ej"], flags=re.IGNORECASE))
                    nombre = f"{metadata['anio']}_{slug(metadata['conv'])}_{ej_slug}"
                else:
                    nombre = f"{slug(ruta_pdf.stem)}_p{n + 1:02d}"

                imagen = ""
                if png:
                    png_folder.mkdir(parents=True, exist_ok=True)
                    destino = png_folder / f"{nombre}.png"
                    i = 2
                    while destino.exists():
                        destino = png_folder / f"{nombre}_{i}.png"
                        i += 1
                    recorte.to_image(resolution=dpi).save(destino)
                    imagen = str(destino.relative_to(output_folder))

                off_x, off_y = float(page.bbox[0]), float(page.bbox[1])
                recortes.append((n, off_x + x0, off_y + page.height - bottom,
                                 off_x + x1, off_y + page.height - top))

                aviso = ""
                if not metadata:
                    aviso = "sin metadatos"
                elif len(cajas) > 1:
                    aviso = f"{len(cajas)} recuadros en la página"

                # asignatura y tema
                asignatura = ruta_pdf.parts[-3].replace(" - Temas", "").replace(" ", "").lower()
                tema = ruta_pdf.stem.split(" - ")[1]

                filas.append({
                    "asignatura": asignatura,
                    "tema": tema,
                    "año": metadata["anio"] if metadata else "",
                    "convocatoria": metadata["conv"].title() if metadata else "",
                    "ejercicio": metadata["ej"] if metadata else "",
                    "enunciado": texto,
                    "imagen": imagen,
                    "archivo": ruta_pdf.name,
                    "aviso": aviso,
                })

    if not recortes:
        return filas, "ningún recuadro detectado"

    # 2) Composición vectorial con pypdf
    # comprobar si existe el directorio de salida y crearlo si no
    (output_folder / "enunciados_PDF").mkdir(parents=True, exist_ok = True)
    salida = output_folder / "enunciados_PDF" / f"{ruta_pdf.stem}_enunciados.pdf"
    writer = PdfWriter()
    hoja, y, num_hoja = None, None, 0
    usados = set()
    reader = PdfReader(ruta_pdf)

    for fila, (n, x0, y0, x1, y1) in zip(filas, recortes):
        # si la página ya se usó (varios recuadros), se relee para no pisar el recorte anterior
        src = (PdfReader(ruta_pdf) if n in usados else reader).pages[n]
        usados.add(n)
        src.mediabox.lower_left = src.cropbox.lower_left = (x0, y0)
        src.mediabox.upper_right = src.cropbox.upper_right = (x1, y1)
        w, h = x1 - x0, y1 - y0

        if uno_por_pagina:
            hoja = writer.add_blank_page(w, h)
            hoja.merge_transformed_page(src, Transformation().translate(-x0, -y0))
            num_hoja += 1
        else:
            escala = min(1.0, (A4[0] - 2 * MARGEN_A4) / w)
            w, h = w * escala, h * escala
            if hoja is None or y - h < MARGEN_A4:
                hoja = writer.add_blank_page(*A4)
                y = A4[1] - MARGEN_A4
                num_hoja += 1
            tx, ty = (A4[0] - w) / 2, y - h
            t = Transformation().translate(-x0, -y0).scale(escala, escala).translate(tx, ty)
            hoja.merge_transformed_page(src, t)
            y -= h + SEPARACION

        # fila["pdf_salida"] = salida.name
        # fila["pagina_salida"] = num_hoja

    with open(salida, "wb") as f:
        writer.write(f)
    return filas, ""

In [8]:
# RUTA_PDF = Path("./pdfplumber/solubilidad/2024 - Solubilidad.pdf")
# #CARPETA_ENTRADA = Path("./pdf_recortados")
# CARPETA_SALIDA = Path("./pdf_recortados")

# CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
# print("¿Existe el PDF de prueba?", RUTA_PDF.exists())

In [9]:
# filas, aviso = procesar_pdf(RUTA_PDF, CARPETA_SALIDA)
# print("Aviso:", aviso or "ninguno")
# pd.DataFrame(filas)

## Procesar carpeta

In [10]:
def process_folder(folder, files_to_process_limit=None):

    # '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio'
    source_folder = Path(folder)
    output_folder_name = "/".join(source_folder.parts[-2:])
    output_folder = Path(f"{root}/cropped_pdf_files/{output_folder_name}")
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Procesando carpeta: {output_folder_name}")

    pdf_files = sorted(p for p in source_folder.rglob("*.pdf") if not p.stem.endswith("_enunciados"))
    if files_to_process_limit is not None:
        pdf_files = pdf_files[:files_to_process_limit]

    print(f"{len(pdf_files)} PDFs a procesar\n")

    todas, problemas = [], []
    for index, file_path in enumerate(pdf_files, start=1):
        try:
            filas, aviso = procesar_pdf(file_path, output_folder)
        except Exception as e:
            problemas.append({"pdf": str(file_path), "problema": f"ERROR: {e}"})
            print(f"[{index}/{len(pdf_files)}] ERROR  {file_path.name}: {e}")
            continue

        todas.extend(filas)

        if aviso:
            problemas.append({"pdf": str(file_path), "problema": aviso})
            print(f"[{index}/{len(pdf_files)}] AVISO  {file_path.name}: {aviso}")
        else:
            print(f"[{index}/{len(pdf_files)}] OK     {file_path.name}: {len(filas)} enunciados")

    # Guardar resultados en CSV
    enunciados_df = pd.DataFrame(todas)
    enunciados_df.to_csv(output_folder / f"{output_folder.name}.csv", index=False)

    if problemas:
        problemas_df = pd.DataFrame(problemas)
        problemas_df.to_csv(output_folder / f"{output_folder.name}_problemas.csv",
                            index=False, encoding="utf-8")

    print("*" * 40)
    print(f"\nProcesamiento completado. Resultados guardados en: {output_folder}\n\n")

## Procesar carpetas

Normalizar texto de nombres de temas y carpetas

In [11]:
import unicodedata

def normalize_text(texto):
    descompuesto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in descompuesto if not unicodedata.combining(c))

Obtener las rutas de los archivos

In [12]:
os.chdir(f"{root}/source_pdf_files")

In [13]:
carpetas_asignaturas = {d.stem:[] for d in Path(".").iterdir() if d.is_dir()}

In [14]:
for asignatura in carpetas_asignaturas:
    carpetas_asignaturas[asignatura] = [d.stem for d in Path(f"./{asignatura}").iterdir() if d.is_dir()]

In [15]:
carpetas_asignaturas

{'Física - Temas': ['campo_gravitatorio',
  'movimiento_ondulatorio',
  'física_cuántica_nuclear',
  'campo_eléctrico_magnético',
  'óptica_geométrica'],
 'Mates CCSS - Temas': ['distribución_binomial',
  'matrices_determinantes',
  'programación_lineal',
  'contraste_hipótesis',
  'sistemas_ecuaciones_lineales_ccss',
  'inferencia_estadística',
  'probabilidad',
  'funciones_ccss'],
 'Química - Temas': ['reactividad_orgánica',
  'redox',
  'enlace_químico',
  'configuración_electrónica',
  'solubilidad',
  'formulación',
  'equilibrio_químico',
  'ácido_base',
  'termoquímica',
  'cantidad_química'],
 'Mates II - Temas': ['funciones',
  'matrices_determinantes',
  'geometría',
  'distribución_normal_binomial',
  'sistemas_de_ecuaciones_lineales',
  'probabilidad',
  'integrales']}

In [16]:
folder_paths = []
for asignatura, carpetas in carpetas_asignaturas.items():
    for carpeta in carpetas:
        folder_paths.append(f"{root}/source_pdf_files/{asignatura}/{carpeta}")

In [17]:
folder_paths[:5]

['/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_gravitatorio',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/movimiento_ondulatorio',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/física_cuántica_nuclear',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/campo_eléctrico_magnético',
 '/home/daniel/code/clasificador_selectividad/source_pdf_files/Física - Temas/óptica_geométrica']

# PROCESAR LOS ARCHIVOS

In [18]:
for folder in folder_paths:
    folder_path = Path(folder)
    process_folder(folder, files_to_process_limit=None)  # Limitar a 5 archivos por carpeta para pruebas

Procesando carpeta: Física - Temas/campo_gravitatorio
11 PDFs a procesar

[1/11] OK     2016 - Campo gravitatorio.pdf: 10 enunciados
[2/11] OK     2017 - Campo gravitatorio.pdf: 12 enunciados
[3/11] OK     2018 - Campo gravitatorio.pdf: 12 enunciados
[4/11] OK     2019 - Campo gravitatorio.pdf: 12 enunciados
[5/11] OK     2020 - Campo gravitatorio.pdf: 12 enunciados
[6/11] OK     2021 - Campo gravitatorio.pdf: 12 enunciados
[7/11] OK     2022 - Campo gravitatorio.pdf: 12 enunciados
[8/11] OK     2023 - Campo gravitatorio.pdf: 12 enunciados
[9/11] OK     2024 - Campo gravitatorio.pdf: 12 enunciados
[10/11] OK     2025 - Campo gravitatorio.pdf: 15 enunciados
[11/11] OK     2026 - Campo gravitatorio.pdf: 2 enunciados
****************************************

Procesamiento completado. Resultados guardados en: /home/daniel/code/clasificador_selectividad/cropped_pdf_files/Física - Temas/campo_gravitatorio


Procesando carpeta: Física - Temas/movimiento_ondulatorio
10 PDFs a procesar

[1/10] 

## Mover archivos .csv

In [19]:
os.chdir(root)

In [20]:
Path("csv_enunciados").mkdir(exist_ok=True)

In [21]:
os.chdir("cropped_pdf_files")

In [24]:
for folder in Path(".").iterdir():
    if folder.is_dir():
        subject = folder.name.split(" - ")[0].replace(" ", "_").lower()
        csv_files = list(folder.rglob("*.csv"))
        for csv_file in csv_files:
            new_filename = f"{subject}_{csv_file.stem}.csv"
            if "_problemas" in csv_file.name:
                Path(f"{root}/csv_enunciados/problemas").mkdir(exist_ok=True)
                output_csv_path = Path(f"{root}/csv_enunciados/problemas/{subject}_{csv_file.stem}_problemas.csv")
            else:
                new_filename = f"{subject}_{csv_file.stem}_enunciados.csv"
                output_csv_path = Path(f"{root}/csv_enunciados/{new_filename}")

            shutil.move(csv_file, output_csv_path)